Loading the parquet file + schema health check

In [1]:
%pip install polars

   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/824.0 kB ? eta -:--:--
   ---

In [2]:
import os
import shutil
import pandas as pd
import matplotlib as plt
import seaborn as sns
import pyarrow.parquet as pq
import fsspec
import time
import duckdb
from pathlib import Path
import polars as pl

In [4]:
urls = {
    "jan": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet",
    "feb": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet",
    "mar": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-03.parquet",
    "apr": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-04.parquet",
    "may": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-05.parquet",
    "jun": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet",
    "jul": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet",
    "aug": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-08.parquet",
    "sep": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-09.parquet",
    "oct": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-10.parquet",
    "nov": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet",
    "dec": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-12.parquet",
}

schemas = {}

# schema health check
for name, url in urls.items():
    with fsspec.open(url, "rb") as f:
        pf = pq.ParquetFile(f)
        schemas[name] = pf.schema_arrow
    print(f"\n=== {name} ====")
    print(pf.schema)

ref_month, ref_schema = next(iter(schemas.items()))
bad = [m for m, s in schemas.items() if not s.equals(ref_schema)]
print("All match" if not bad else f" mismatch: {bad}")


=== jan ====
required group field_id=-1 schema {
  optional int32 field_id=-1 VendorID;
  optional int64 field_id=-1 tpep_pickup_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 tpep_dropoff_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 passenger_count;
  optional double field_id=-1 trip_distance;
  optional int64 field_id=-1 RatecodeID;
  optional binary field_id=-1 store_and_fwd_flag (String);
  optional int32 field_id=-1 PULocationID;
  optional int32 field_id=-1 DOLocationID;
  optional int64 field_id=-1 payment_type;
  optional double field_id=-1 fare_amount;
  optional double field_id=-1 extra;
  optional double field_id=-1 mta_tax;
  optional double field_id=-1 tip_amount;
  optional double field_id=-1 tolls_amount;
  optional double field_id=-1 improvement_s

One-time local combined parquet build

In [ ]:
local_parquet = Path("data/raw/yellow_2025_combined.parquet")
local_monthly_dir = Path("data/raw/monthly_2025")
DOWNLOAD_BUFFER_SIZE = 8 * 1024 * 1024
ROW_GROUP_SIZE = 500000

def materialize_monthly_parquets(urls_dict, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)

    local_paths = []
    t0 = time.perf_counter()

    for url in urls_dict.values():
        filename = url.rsplit("/", 1)[-1]
        target = local_dir / filename

        if not target.exists():
            tmp_target = target.with_suffix(target.suffix + ".tmp")

            try:
                with fsspec.open(url, "rb") as src, open(tmp_target, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=DOWNLOAD_BUFFER_SIZE)
                os.replace(tmp_target, target)
            except Exception:
                tmp_target.unlink(missing_ok=True)
                raise

        local_paths.append(target.as_posix())

    t1 = time.perf_counter()
    print(f"Monthly cache ready in {t1 - t0:.2f}s -> {local_dir}")
    return local_paths

def build_local_combined_parquet(urls_dict, output_path, monthly_dir):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists():
        print(f"Already exists: {output_path}")
        return

    monthly_paths = materialize_monthly_parquets(urls_dict, monthly_dir)

    quoted_paths = ", ".join([f"'{p}'" for p in monthly_paths])

    t0 = time.perf_counter()
    # 500k balances build throughput and downstream scan granularity without large memory spikes during write.
    sql = f"""
    COPY (
        SELECT *
        FROM read_parquet([{quoted_paths}], union_by_name=false)
    )
    TO '{output_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE {ROW_GROUP_SIZE});
    """

    with duckdb.connect() as con:
        con.execute(f"PRAGMA threads={max(1, os.cpu_count() or 1)}")
        con.execute("SET preserve_insertion_order = false")
        con.execute(sql)

    t1 = time.perf_counter()
    print(f"Built once in {t1 - t0:.2f}s -> {output_path}")

build_local_combined_parquet(urls, local_parquet, local_monthly_dir)


Monthly cache ready in 1352.89s -> data\raw\monthly_2025
Built once in 18.92s -> data\raw\yellow_2025_combined.parquet


In [4]:
yellow_df = pl.read_parquet(source="data/raw/yellow_2025_combined.parquet")
yellow_df.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,1,"""N""",164,233,1,7.9,0.0,0.5,1.0,0.0,1.0,13.65,2.5,0.0,0.75
2,2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,1,"""N""",229,138,2,42.9,5.0,0.5,0.0,6.94,1.0,59.59,2.5,0.0,0.75
1,2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,1,"""N""",79,162,1,14.9,3.25,0.5,1.0,0.0,1.0,20.65,2.5,0.0,0.75
1,2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,1,"""N""",263,163,1,11.4,3.25,0.5,3.2,0.0,1.0,19.35,2.5,0.0,0.75
2,2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,1,"""N""",224,186,1,12.1,0.0,0.5,1.15,0.0,1.0,18.0,2.5,0.0,0.75


In [5]:
yellow_df.describe().round(2)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
count,48722602.00,48722602,48722602,37110708.00,48722602.00,37110708.00,48722602.0,48722602.00,48722602.00,48722602.00,48722602.00,48722602.00,48722602.00,48722602.00,48722602.00,48722602.00,37110708.00,37110708.00,48722602.00
mean,1.86,2025-07-06 05:22:36.621950,2025-07-06 05:39:57.973504,1.30,6.84,3.23,161.5,161.11,0.94,18.42,1.16,0.48,2.84,0.50,0.95,26.91,2.18,0.15,0.53
min,1.00,2007-12-05 18:45:00,2007-12-05 19:02:00,0.00,0.00,1.00,1.0,1.00,0.00,-1807.60,-17.39,-21.74,-333.33,-148.17,-1.00,-1832.85,-2.50,-1.75,-0.75
25%,2.00,2025-04-08 14:04:38,2025-04-08 14:21:56,1.00,1.04,1.00,114.0,107.00,1.00,8.60,0.00,0.50,0.00,0.00,1.00,15.73,2.50,0.00,0.00
50%,2.00,2025-07-03 12:25:13,2025-07-03 12:42:35,1.00,1.85,1.00,161.0,162.00,1.00,13.58,0.00,0.50,2.00,0.00,1.00,21.35,2.50,0.00,0.75
75%,2.00,2025-10-06 13:54:53.750000,2025-10-06 14:13:21.750000,1.00,3.70,1.00,233.0,233.00,1.00,22.60,2.50,0.50,3.99,0.00,1.00,30.75,2.50,0.00,0.75
max,7.00,2025-12-31 23:59:59,2026-01-05 13:00:34,9.00,397994.37,99.00,265.0,265.00,5.00,863372.12,133.60,5243.38,960.94,916.87,2.50,863380.37,2.50,6.75,1.75
std,0.68,NaN,NaN,0.73,649.36,14.27,66.2,70.45,0.75,142.85,1.82,0.76,3.97,2.12,0.28,143.48,0.95,0.53,0.36


Interesting... what has caught my eye is that I am seeing 2007 somewhere and now I am worried I got some wrong data yet I am certain that I got the right one.... even the links show the date... I am also seeing negatives... I need to find out how to work with these values!

In [3]:
exists_2007 = (yellow_df["tpep_pickup_datetime"].dt.year == 2007).any()
exists_2007

np.True_

Oh dear... now this needs me to pause and figure out where this 2007 is coming from

In [5]:
filter_2007 = yellow_df[(yellow_df["tpep_pickup_datetime"].dt.year == 2007)]
filter_2007

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
8855397,2,2007-12-05 18:45:00,2007-12-05 19:02:00,1.0,3.0,1.0,N,142,234,2,17.0,1.0,0.5,0.0,0.0,1.0,22.75,2.5,0.0,0.75


God knows how that row snuck in there! Super random! There may be more though... lemme figure out which years are used in the whole column

In [3]:
unique_years = yellow_df["tpep_pickup_datetime"].dt.year.unique()
unique_years

array([2025, 2009, 2024, 2007, 2008], dtype=int32)

Great... there's more of them! Meanwhile my conda kernel can't handle this workload... it keeps crashing so I may need to switch to something else soon! Or let's try polars

In [3]:
filter_2008 = yellow_df[(yellow_df["tpep_pickup_datetime"].dt.year == 2008)]
filter_2009 = yellow_df[(yellow_df["tpep_pickup_datetime"].dt.year == 2009)]
filter_2024 = yellow_df[(yellow_df["tpep_pickup_datetime"].dt.year == 2024)]

In [4]:
filter_2008

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
47625785,2,2008-12-31 23:04:21,2008-12-31 23:32:25,1.0,18.12,2.0,N,132,238,2,70.0,0.0,0.5,0.0,6.94,1.0,80.94,2.5,0.0,0.0


In [5]:
filter_2009

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2739330,2,2009-01-01 00:19:34,2009-01-01 01:10:21,6.0,10.77,1.0,N,138,239,2,52.7,5.0,0.5,0.00,6.94,1.0,70.39,2.5,1.75,0.00
14148392,2,2009-01-01 00:20:39,2009-01-01 00:20:49,5.0,4.47,1.0,N,230,261,1,24.0,1.0,0.5,2.98,0.00,1.0,32.73,2.5,0.00,0.75
17885742,2,2009-01-01 08:52:26,2009-01-01 10:00:26,1.0,11.95,1.0,N,138,163,1,64.6,0.0,0.5,16.65,13.88,1.0,99.88,2.5,0.00,0.75
37183565,2,2009-01-01 12:52:15,2009-01-01 13:12:15,1.0,2.13,1.0,N,43,230,1,18.4,0.0,0.5,4.63,0.00,1.0,27.78,2.5,0.00,0.75
44795947,2,2009-01-01 14:08:04,2009-01-01 14:45:04,1.0,11.56,1.0,N,138,142,2,50.6,0.0,0.5,0.00,13.88,1.0,68.48,2.5,0.00,0.00
47626534,2,2009-01-01 00:05:50,2009-01-01 00:10:56,1.0,1.26,1.0,N,239,50,2,8.6,1.0,0.5,0.00,0.00,1.0,14.35,2.5,0.00,0.75


In [6]:
filter_2024

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
5017662,2,2024-12-31 23:30:03,2024-12-31 23:43:02,1.0,3.00,1.0,N,246,13,1,16.3,1.0,0.5,5.32,0.0,1.0,26.62,2.5,0.00,0.0
5017744,2,2024-12-31 23:31:38,2024-12-31 23:41:48,1.0,1.03,1.0,N,43,140,2,10.7,1.0,0.5,0.00,0.0,1.0,15.70,2.5,0.00,0.0
5017745,2,2024-12-31 23:46:38,2025-01-01 00:03:03,1.0,3.95,1.0,N,229,24,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.00,0.0
5017918,2,2024-12-31 23:56:19,2025-01-01 00:11:19,6.0,2.28,1.0,N,68,107,1,14.9,1.0,0.5,3.98,0.0,1.0,23.88,2.5,0.00,0.0
5018165,2,2024-12-31 23:55:37,2025-01-01 00:01:26,1.0,1.12,1.0,N,56,56,2,7.9,1.0,0.5,0.00,0.0,1.0,10.40,0.0,0.00,0.0
5018369,2,2024-12-31 23:52:40,2025-01-01 00:23:03,1.0,6.72,1.0,N,114,151,1,34.5,1.0,0.5,7.90,0.0,1.0,47.40,2.5,0.00,0.0
5019333,2,2024-12-31 23:49:24,2024-12-31 23:57:30,1.0,2.44,1.0,N,142,151,1,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.00,0.0
5020998,2,2024-12-31 23:27:13,2024-12-31 23:35:48,1.0,1.53,1.0,N,170,141,1,10.0,1.0,0.5,3.00,0.0,1.0,18.00,2.5,0.00,0.0
5020999,2,2024-12-31 23:37:42,2024-12-31 23:43:10,1.0,0.92,1.0,N,229,141,1,7.2,1.0,0.5,1.83,0.0,1.0,14.03,2.5,0.00,0.0
5021381,2,2024-12-31 23:51:20,2025-01-01 00:00:00,1.0,12.89,1.0,N,138,40,1,48.5,6.0,0.5,11.55,0.0,1.0,69.30,0.0,1.75,0.0


Well this is manageable... I'll just need to remove the rows based on the filters I have made... However interestingly, I hadn't considered/thought about the edge case of someone travelling when it's just before the new year and hence they end up elsewhere in another year... those ones I guess I may have to think it through... Alright I contacted DK and so I'll just delete the overlaps, and ignore the fare column becuase after all... it may not be a predictor

In [5]:
yellow_df_cleaned = yellow_df.remove((yellow_df["tpep_pickup_datetime"].dt.year == 2007) &(yellow_df["tpep_pickup_datetime"].dt.year == 2008) & (yellow_df["tpep_pickup_datetime"].dt.year == 2009) & (yellow_df["tpep_pickup_datetime"].dt.year == 2024))

In [6]:
yellow_df_cleaned.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2025-03-31 08:25:18,2025-03-31 08:31:33,1,0.92,1,"""N""",164,233,1,7.9,0.0,0.5,1.0,0.0,1.0,13.65,2.5,0.0,0.75
2,2025-03-31 08:35:17,2025-03-31 08:56:30,1,10.55,1,"""N""",229,138,2,42.9,5.0,0.5,0.0,6.94,1.0,59.59,2.5,0.0,0.75
1,2025-03-31 08:45:27,2025-03-31 08:59:43,1,2.8,1,"""N""",79,162,1,14.9,3.25,0.5,1.0,0.0,1.0,20.65,2.5,0.0,0.75
1,2025-03-31 08:12:55,2025-03-31 08:21:41,1,1.5,1,"""N""",263,163,1,11.4,3.25,0.5,3.2,0.0,1.0,19.35,2.5,0.0,0.75
2,2025-03-31 08:21:32,2025-03-31 08:33:03,1,1.57,1,"""N""",224,186,1,12.1,0.0,0.5,1.15,0.0,1.0,18.0,2.5,0.0,0.75


Let's now check the stats of our cleaned df...

In [8]:
yellow_df_cleaned.describe()

: 

In [ ]:
yellow_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48722602 entries, 0 to 48722601
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64 